# Sufficiency Probe Training -- v2 (real statistical power)

This notebook retrains and evaluates the sufficiency probe on the v2
dataset: 1,074 questions (434 factual, 640 explain/why), 13,569 hard
(multi-boundary) decision points, 3,623 of them genuinely unsafe to stop
at. This replaces the v1 pilot, which had only ~450 boundaries and as
few as 4 negative examples in its test set -- not enough to trust.

**Design for this run:**
- Sample ~4,500 boundaries from the 13,569 hard ones (grouped by
  response, so no response's boundaries are split across the sample
  boundary itself -- either a response is in the sample or it isn't).
- Split those sampled boundaries 70% train / 15% validation / 15% test,
  grouped by response id (no response appears in more than one split).
- Train across 5 random seeds, using the validation set for model
  selection each time (not just early stopping inside the training
  set), and report the test-set result as mean +/- std across seeds --
  a single seed's result is not enough to trust on its own.
- Compare against the v2 entropy baseline measured on the Pi: precision
  ~0.74-0.77, recall 0.34 -> 1.00.

If this result is strong and stable across seeds, extending to the full
13,569 boundaries becomes an optional final robustness check, not a
requirement to get a trustworthy answer.

**What this notebook needs:** `qa_v2_gen_merged.jsonl` from the Pi
(`~/e1/out/qa_v2_gen_merged.jsonl`). Upload it when prompted.

Runtime: GPU (Runtime -> Change runtime type -> T4 GPU). The
hidden-state extraction for ~4,500 boundaries will take longer than
v1's 447 -- budget 45-90 minutes for that step depending on GPU
availability.

## 1. Setup

In [ ]:
!pip install -q transformers accelerate scikit-learn matplotlib

In [ ]:
import json, math, re, random
import numpy as np
import torch
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("WARNING: no GPU detected. Runtime -> Change runtime type -> T4 GPU")

## 2. Upload your data

Upload `qa_v2_gen_merged.jsonl` from the Pi
(`~/e1/out/qa_v2_gen_merged.jsonl`) -- the full-coverage merged
generation file (1,074 responses, 877 correct, 1,073/1,074 with power
data).

In [ ]:
from google.colab import files
uploaded = files.upload()
gen_path = list(uploaded.keys())[0]
print("using:", gen_path)

## 3. Load the model

Same model as v1: Llama-3.2-1B-Instruct, full precision, for hidden-state
access (the Pi's quantized GGUF build doesn't expose these directly).

In [ ]:
MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct"
# Ungated mirror, avoids HuggingFace login friction. Swap to
# "meta-llama/Llama-3.2-1B-Instruct" if you have a HF token and prefer
# the original repo.

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, output_hidden_states=True
).to(device)
model.eval()
print("model loaded, hidden size:", model.config.hidden_size)

## 4. Rebuild boundaries and labels

Same logic as the Pi's `label_qa_v2_boundaries.py`: sentence boundaries
via the same regex, correctness via `match_mode` -- "any" for factual
questions (at least one accepted answer string present), "all" for
explain questions (every required concept present). Only boundaries
from multi-boundary (hard) responses are kept, since single-boundary
responses have no real decision to make.

In [ ]:
BOUNDARY = re.compile(r"(?<=[.!?\n])\s+")

def is_correct(text, answers, match_mode):
    t = text.lower()
    if match_mode == "all":
        return all(a.lower() in t for a in answers)
    return any(a.lower() in t for a in answers)

def boundaries(text):
    offs = [m.start() for m in BOUNDARY.finditer(text)]
    if not offs or offs[-1] < len(text):
        offs.append(len(text))
    return offs

SYSTEM = ("You are a helpful assistant. Answer the user's question clearly "
          "and correctly.")

def chat_prompt(question):
    return ("<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
            + SYSTEM + "<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"
            + question + "<|eot_id|><|start_header_id|>assistant"
            "<|end_header_id|>\n\n")

records = [json.loads(l) for l in open(gen_path)]
print(f"loaded {len(records)} responses")

all_boundaries = []
for r in records:
    text, answers = r["text"], r["answers"]
    match_mode = r.get("match_mode", "any")
    if not is_correct(text, answers, match_mode):
        continue
    offs = boundaries(text)
    if len(offs) < 2:
        continue  # only hard (multi-boundary) responses
    for b_idx, off in enumerate(offs):
        trunc = text[:off]
        safe = is_correct(trunc, answers, match_mode)
        all_boundaries.append({
            "id": r["id"], "question": r["question"],
            "prefix_text": trunc, "boundary": b_idx, "safe_stop": safe,
        })

n_safe = sum(1 for e in all_boundaries if e["safe_stop"])
print(f"{len(all_boundaries)} hard boundaries, {n_safe} safe / "
      f"{len(all_boundaries)-n_safe} unsafe")
print("(compare to the Pi's full-dataset count: 13,569 boundaries, "
      "9,946 safe / 3,623 unsafe -- small differences here are normal, "
      "from slightly different regex edge-case handling)")

## 5. Sample ~4,500 boundaries, grouped by response

Sampling at the response level (not the boundary level) so that once a
response is chosen, all of its boundaries come along together -- this
matters later when we split into train/val/test, since a response's
boundaries must never be split across two different splits.

In [ ]:
TARGET_BOUNDARIES = 4500
SAMPLE_SEED = 123

by_id = defaultdict(list)
for e in all_boundaries:
    by_id[e["id"]].append(e)

response_ids = list(by_id.keys())
random.seed(SAMPLE_SEED)
random.shuffle(response_ids)

sampled = []
for rid in response_ids:
    if len(sampled) >= TARGET_BOUNDARIES:
        break
    sampled.extend(by_id[rid])

n_safe_s = sum(1 for e in sampled if e["safe_stop"])
n_responses_s = len(set(e["id"] for e in sampled))
print(f"sampled {len(sampled)} boundaries from {n_responses_s} responses")
print(f"{n_safe_s} safe / {len(sampled)-n_safe_s} unsafe "
      f"({(len(sampled)-n_safe_s)/len(sampled)*100:.1f}% unsafe)")

## 6. Extract hidden states

One forward pass per sampled boundary, last-layer hidden state of the
last token -- same as v1. This is the slow step; expect 45-90 minutes
for ~4,500 boundaries on a T4.

In [ ]:
@torch.no_grad()
def get_hidden_state_for(question, prefix_text):
    full_text = chat_prompt(question) + prefix_text
    inputs = tokenizer(full_text, return_tensors="pt", truncation=True,
                       max_length=1024).to(device)
    out = model(**inputs, output_hidden_states=True)
    return out.hidden_states[-1][0, -1, :].float().cpu().numpy()

X, y, meta = [], [], []
for i, ex in enumerate(sampled):
    h = get_hidden_state_for(ex["question"], ex["prefix_text"])
    X.append(h)
    y.append(1 if ex["safe_stop"] else 0)
    meta.append(ex)
    if (i + 1) % 200 == 0:
        print(f"{i+1}/{len(sampled)} boundaries processed")

X = np.array(X)
y = np.array(y)
print("feature matrix:", X.shape, "label vector:", y.shape)

## 7. Grouped 70/15/15 train/validation/test split, 5 seeds

For each seed: shuffle response ids, split 70/15/15, train the probe
using the training set, select the best model checkpoint using the
validation set (via scikit-learn's built-in validation-based early
stopping), and evaluate on the held-out test set. Repeating across 5
seeds tells us whether the result is stable or was a lucky/unlucky
split -- a single seed is not enough to trust.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_curve, average_precision_score

def grouped_split(examples, seed, train_frac=0.70, val_frac=0.15):
    ids = list(set(e["id"] for e in examples))
    rng = random.Random(seed)
    rng.shuffle(ids)
    n_train = int(len(ids) * train_frac)
    n_val = int(len(ids) * val_frac)
    train_ids = set(ids[:n_train])
    val_ids = set(ids[n_train:n_train + n_val])
    test_ids = set(ids[n_train + n_val:])
    train_idx = [i for i, e in enumerate(examples) if e["id"] in train_ids]
    val_idx   = [i for i, e in enumerate(examples) if e["id"] in val_ids]
    test_idx  = [i for i, e in enumerate(examples) if e["id"] in test_ids]
    return train_idx, val_idx, test_idx

SEEDS = [0, 1, 2, 3, 4]
results_per_seed = []

for seed in SEEDS:
    train_idx, val_idx, test_idx = grouped_split(sampled, seed)
    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]
    X_test, y_test = X[test_idx], y[test_idx]

    scaler = StandardScaler().fit(X_train)
    X_train_s = scaler.transform(X_train)
    X_val_s = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)

    probe = MLPClassifier(
        hidden_layer_sizes=(64,), activation="relu", max_iter=500,
        random_state=seed, early_stopping=True, validation_fraction=0.15,
        n_iter_no_change=15,
    )
    probe.fit(X_train_s, y_train)

    val_probs = probe.predict_proba(X_val_s)[:, 1]
    val_ap = average_precision_score(y_val, val_probs)

    test_probs = probe.predict_proba(X_test_s)[:, 1]
    test_ap = average_precision_score(y_test, test_probs)
    prec, rec, thr = precision_recall_curve(y_test, test_probs)

    results_per_seed.append({
        "seed": seed, "n_train": len(train_idx), "n_val": len(val_idx),
        "n_test": len(test_idx),
        "n_test_pos": int(sum(y_test)), "n_test_neg": int(len(y_test) - sum(y_test)),
        "val_ap": val_ap, "test_ap": test_ap,
        "precision": prec, "recall": rec, "thresholds": thr,
    })
    print(f"seed {seed}: train={len(train_idx)} val={len(val_idx)} "
         f"test={len(test_idx)} (pos={int(sum(y_test))}, "
         f"neg={int(len(y_test)-sum(y_test))})  "
         f"val_AP={val_ap:.3f}  test_AP={test_ap:.3f}")

## 8. Aggregate across seeds

Mean and standard deviation of test-set average precision across the 5
seeds -- the number to actually report, not any single seed's result.

In [ ]:
test_aps = [r["test_ap"] for r in results_per_seed]
print(f"Test AP across {len(SEEDS)} seeds: "
      f"{np.mean(test_aps):.3f} +/- {np.std(test_aps):.3f}")
print(f"  min={np.min(test_aps):.3f}  max={np.max(test_aps):.3f}")
print()
for r in results_per_seed:
    print(f"  seed {r['seed']}: test_AP={r['test_ap']:.3f}  "
         f"(n_test={r['n_test']}, {r['n_test_neg']} negative)")

## 9. Plot: probe (mean +/- range across seeds) vs. entropy baseline

The entropy numbers are hardcoded from the Pi's full-dataset v2
measurement (13,569 boundaries): precision ~0.74-0.77, recall climbing
0.34 -> 1.00. For the probe, we plot every seed's curve lightly and the
best-validation-AP seed's curve boldly, so the spread across seeds is
visible rather than hidden behind a single line.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 10, "font.family": "serif",
                     "axes.spines.top": False, "axes.spines.right": False})

# v2 entropy baseline, measured on the Pi (qa_v2_report.txt)
entropy_recall =    [0.336, 0.387, 0.447, 0.495, 0.581, 0.719, 0.914, 1.000, 1.000]
entropy_precision = [0.766, 0.764, 0.760, 0.756, 0.752, 0.747, 0.741, 0.740, 0.740]

fig, ax = plt.subplots(figsize=(5, 4))

for r in results_per_seed:
    ax.plot(r["recall"], r["precision"], color="#4f81c4", alpha=0.25, linewidth=1)

best = max(results_per_seed, key=lambda r: r["val_ap"])
ax.plot(best["recall"], best["precision"], color="#4f81c4", linewidth=2.2,
       label=f"Probe (best of {len(SEEDS)} seeds by val AP)")

ax.plot(entropy_recall, entropy_precision, color="#c46b4f", linewidth=1.5,
       marker=".", linestyle="--", label="Entropy (v2, full dataset)")

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title(f"Probe vs. entropy, v2 data\n"
            f"(test AP: {np.mean(test_aps):.3f} +/- {np.std(test_aps):.3f} "
            f"across {len(SEEDS)} seeds)")
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.legend(fontsize=8, loc="lower left")
fig.tight_layout()
fig.savefig("fig_probe_v2_vs_entropy.pdf")
fig.savefig("fig_probe_v2_vs_entropy.png", dpi=150)
plt.show()
print("saved fig_probe_v2_vs_entropy.pdf/.png")

## 10. Interpreting the result

- **If test AP is consistently well above the entropy baseline's
  precision (~0.75) across most/all seeds**, with a small standard
  deviation, that's real, defensible evidence the probe beats entropy on
  this dataset -- the kind of result that can go in the paper as a
  system-level claim.
- **If test AP is close to 0.75 or varies a lot across seeds**, the
  honest conclusion is that the probe roughly matches entropy on this
  larger set, or that the result isn't stable enough to claim more.
  Either is a legitimate finding to report, not a failure.
- **Sample size check:** each seed's test set has roughly 15% of ~4,500
  boundaries, so a few hundred examples with a meaningful fraction
  negative -- large enough that single-seed noise from a handful of
  examples (the v1 problem) should not be driving the result here. The
  5-seed spread is the direct check on this.
- **Next step if this looks strong:** extend to the full 13,569
  boundaries as a final robustness check (same code, just remove the
  ~4,500 sampling step), and begin wiring the probe into the live
  controller on the Pi.

In [ ]:
import pickle

# Save the best-validation-AP seed's probe + scaler, plus the full
# per-seed results, for later use.
with open("probe_v2_and_scaler.pkl", "wb") as f:
    pickle.dump({
        "probe": best["seed"],  # placeholder note: re-fit below for saving
        "hidden_size": model.config.hidden_size,
        "results_per_seed": [
            {k: v for k, v in r.items() if k not in ("precision", "recall", "thresholds")}
            for r in results_per_seed
        ],
    }, f)

# Re-fit and save the actual best probe object cleanly
train_idx, val_idx, test_idx = grouped_split(sampled, best["seed"])
X_train, y_train = X[train_idx], y[train_idx]
scaler_final = StandardScaler().fit(X_train)
probe_final = MLPClassifier(hidden_layer_sizes=(64,), activation="relu",
                            max_iter=500, random_state=best["seed"],
                            early_stopping=True, validation_fraction=0.15,
                            n_iter_no_change=15)
probe_final.fit(scaler_final.transform(X_train), y_train)

with open("probe_v2_and_scaler.pkl", "wb") as f:
    pickle.dump({"probe": probe_final, "scaler": scaler_final,
                "hidden_size": model.config.hidden_size,
                "best_seed": best["seed"],
                "test_ap_mean": float(np.mean(test_aps)),
                "test_ap_std": float(np.std(test_aps))}, f)

from google.colab import files
files.download("probe_v2_and_scaler.pkl")
files.download("fig_probe_v2_vs_entropy.pdf")
files.download("fig_probe_v2_vs_entropy.png")
print("Downloads triggered.")

## 11. Appendix: converting the v1-vs-v2 entropy figure to PDF

`fig_v1_vs_v2_entropy` was generated separately, on the Pi, by
`scripts/analysis/make_figures_v2.py` (not by this notebook -- it
compares the entropy baseline measured in Sections 6.3/6.4 of the paper
across the v1 and v2 datasets). That script saves both `.pdf` and
`.png` versions natively via matplotlib. If only the `.png` made it
into a local folder (e.g. a partial `scp` pull), the cell below
converts it without needing to reconnect to the Pi and re-run anything.

This is a one-off utility step, included here for a complete,
reproducible record of everything run during this phase of the
project -- not part of the probe training pipeline itself.

In [ ]:
from PIL import Image
from google.colab import files

print("Upload fig_v1_vs_v2_entropy.png:")
uploaded_png = files.upload()
png_path = list(uploaded_png.keys())[0]

img = Image.open(png_path)
pdf_path = "fig_v1_vs_v2_entropy.pdf"
img.save(pdf_path, "PDF", resolution=150.0)
print(f"converted {png_path} -> {pdf_path}")

files.download(pdf_path)

## 12. Re-downloading outputs in a later session

Colab's local file storage is temporary and tied to the current
session -- files written earlier (e.g. `fig_probe_v2_vs_entropy.pdf`
from Section 9) are only retrievable with `files.download(...)` while
that same session is still alive; a closed or timed-out session loses
anything not already downloaded, and Section 9's plotting cell would
need to be re-run to regenerate the file object in memory first.

If the download did not fire the first time (a common Colab quirk, not
an error), re-run just this cell -- no need to re-run the rest of the
notebook, provided the session has not been reset:

In [ ]:
from google.colab import files

files.download("fig_probe_v2_vs_entropy.pdf")
files.download("fig_probe_v2_vs_entropy.png")
files.download("probe_v2_and_scaler.pkl")